In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "aerolineas_dev")
dbutils.widgets.text("source", "silver")
dbutils.widgets.text("sink", "golden")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

In [0]:
schema_source = "silver"
schema_sink = "golden"

target_table_aerolinea = (
    f"{catalog}.{schema_sink}.kpi_aerolinea_mes"
)

target_table_ruta = (
    f"{catalog}.{schema_sink}.kpi_ruta_mes"
)

#  ---  Mantener el calendario mensual en UTC  --- 
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_aerolineas = spark.table(
    f"{catalog}.{schema_source}.tbl_dim_aerolineas"
).select(
    "aerolinea_id",
    "nombre_aerolinea",
    "pais_sede"
)

df_aeropuertos = spark.table(
    f"{catalog}.{schema_source}.tbl_dim_aeropuertos"
).select(
    "aeropuerto_iata",
    "nombre_aeropuerto",
    "ciudad",
    "pais"
)

df_rutas = spark.table(
    f"{catalog}.{schema_source}.tbl_dim_rutas"
).select(
    "ruta_id",
    "origen_iata",
    "destino_iata",
    "tipo_ruta",
    "distancia_km"
)

df_vuelos = spark.table(
    f"{catalog}.{schema_source}.tbl_fact_vuelos"
).select(
    "vuelo_id",
    "aerolinea_id",
    "ruta_id",
    "fecha_salida_programada_utc",
    "estado_vuelo",
    "demora_salida_min",
    "demora_llegada_min"
)

df_pasajeros = spark.table(
    f"{catalog}.{schema_source}.tbl_fact_pasajeros"
).select(
    "vuelo_id",
    "pasajeros_con_asiento",
    "pasajeros_total_a_bordo",
    "asientos_disponibles",
    "pasajeros_no_show"
)

df_combustible = spark.table(
    f"{catalog}.{schema_source}.tbl_fact_combustible"
).select(
    "vuelo_id",
    "combustible_cargado_l",
    "combustible_consumido_l",
    "costo_carga_usd"
)

df_ventas = spark.table(
    f"{catalog}.{schema_source}.tbl_fact_ventas_boletos"
).select(
    "venta_id",
    "vuelo_id",
    "ingreso_neto_usd"
)

df_equipajes = spark.table(
    f"{catalog}.{schema_source}.tbl_fact_equipajes"
).select(
    "vuelo_id",
    "ingreso_equipaje_usd",
    "piezas_facturadas",
    "piezas_extraviadas",
    "piezas_danadas"
)

In [0]:
def validar_clave(df, clave, nombre):
    invalidos = df.filter(
        F.col(clave).isNull()
        | (F.trim(F.col(clave)) == "")
    )

    if invalidos.limit(1).count() > 0:
        display(invalidos)
        raise ValueError(f"{nombre}: clave vacía o nula.")

    duplicados = (
        df.groupBy(clave)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicados.limit(1).count() > 0:
        display(duplicados)
        raise ValueError(f"{nombre}: claves duplicadas.")


for df, clave, nombre in [
    (df_aerolineas, "aerolinea_id", "aerolineas"),
    (df_aeropuertos, "aeropuerto_iata", "aeropuertos"),
    (df_rutas, "ruta_id", "rutas"),
    (df_vuelos, "vuelo_id", "vuelos"),
    (df_pasajeros, "vuelo_id", "pasajeros"),
    (df_combustible, "vuelo_id", "combustible"),
    (df_ventas, "venta_id", "ventas"),
    (df_equipajes, "vuelo_id", "equipajes")
]:
    validar_clave(df, clave, nombre)


#  --- Evitar reemplazar Gold con una carga vacía  --- 
if df_vuelos.limit(1).count() == 0:
    raise ValueError("Silver vuelos está vacío. Se detiene la carga Gold.")

vuelos_invalidos = df_vuelos.filter(
    F.col("fecha_salida_programada_utc").isNull()
    | F.col("estado_vuelo").isNull()
    | ~F.col("estado_vuelo").isin(
        "REALIZADO", "DEMORADO", "CANCELADO"
    )
)

if vuelos_invalidos.limit(1).count() > 0:
    display(vuelos_invalidos)
    raise ValueError("Existen vuelos sin fecha o con estado inválido.")

In [0]:
df_ventas_por_vuelo = (
    df_ventas
    .groupBy("vuelo_id")
    .agg(
        F.sum("ingreso_neto_usd").alias("ingreso_neto_boletos_usd")
    )
)


def validar_cobertura(df, columna, referencia, clave, nombre):
    claves_referencia = referencia.select(
        F.col(clave).alias(columna)
    ).distinct()

    faltantes = df.join(
        claves_referencia,
        on=columna,
        how="left_anti"
    )

    if faltantes.limit(1).count() > 0:
        display(faltantes)
        raise ValueError(f"{nombre}: faltan registros relacionados.")


validar_cobertura(
    df_vuelos, "aerolinea_id",
    df_aerolineas, "aerolinea_id",
    "Vuelos → aerolíneas"
)

validar_cobertura(
    df_vuelos, "ruta_id",
    df_rutas, "ruta_id",
    "Vuelos → rutas"
)

validar_cobertura(
    df_rutas, "origen_iata",
    df_aeropuertos, "aeropuerto_iata",
    "Rutas → aeropuerto origen"
)

validar_cobertura(
    df_rutas, "destino_iata",
    df_aeropuertos, "aeropuerto_iata",
    "Rutas → aeropuerto destino"
)

#  --- En tus CSV completos esperamos datos relacionados para cada vuelo  --- 
for df, nombre in [
    (df_pasajeros, "pasajeros"),
    (df_combustible, "combustible"),
    (df_ventas_por_vuelo, "ventas"),
    (df_equipajes, "equipajes")
]:
    validar_cobertura(
        df_vuelos, "vuelo_id",
        df, "vuelo_id",
        f"Vuelos → {nombre}"
    )

    validar_cobertura(
        df, "vuelo_id",
        df_vuelos, "vuelo_id",
        f"{nombre} → vuelos"
    )

In [0]:
df_origen = df_aeropuertos.select(
    F.col("aeropuerto_iata").alias("origen_iata"),
    F.col("nombre_aeropuerto").alias("nombre_aeropuerto_origen"),
    F.col("ciudad").alias("ciudad_origen"),
    F.col("pais").alias("pais_origen")
)

df_destino = df_aeropuertos.select(
    F.col("aeropuerto_iata").alias("destino_iata"),
    F.col("nombre_aeropuerto").alias("nombre_aeropuerto_destino"),
    F.col("ciudad").alias("ciudad_destino"),
    F.col("pais").alias("pais_destino")
)

df_base = (
    df_vuelos
    .join(df_aerolineas, "aerolinea_id", "inner")
    .join(df_rutas, "ruta_id", "inner")
    .join(df_origen, "origen_iata", "inner")
    .join(df_destino, "destino_iata", "inner")
    .join(df_pasajeros, "vuelo_id", "inner")
    .join(df_combustible, "vuelo_id", "inner")
    .join(df_ventas_por_vuelo, "vuelo_id", "inner")
    .join(df_equipajes, "vuelo_id", "inner")
    .withColumn(
        "periodo",
        F.trunc(F.col("fecha_salida_programada_utc").cast("date"), "month")
    )
    .withColumn("anio", F.year("periodo"))
    .withColumn("mes", F.month("periodo"))
    .withColumn(
        "es_operado",
        F.col("estado_vuelo").isin("REALIZADO", "DEMORADO")
    )
    .withColumn(
        "es_cancelado",
        F.col("estado_vuelo") == "CANCELADO"
    )
    .withColumn(
        "es_puntual",
        F.col("es_operado") & (F.col("demora_salida_min") <= 15)
    )
)

validar_clave(df_base, "vuelo_id", "Base Gold")

if df_base.count() != df_vuelos.count():
    raise ValueError("La unión cambió la cantidad de vuelos.")

In [0]:
def division_segura(numerador, denominador):
    return (
        F.col(numerador)
        / F.when(
            F.col(denominador) > 0,
            F.col(denominador)
        )
    )


def sumar_operados(columna, alias):
    return F.sum(
        F.when(F.col("es_operado"), F.col(columna))
        .otherwise(F.lit(0))
    ).alias(alias)


def agregar_indicadores(df, claves):
    return (
        df.groupBy(*claves)
        .agg(
            F.count("vuelo_id").alias("vuelos_programados"),
            F.sum(F.col("es_operado").cast("long")).alias("vuelos_operados"),
            F.sum(F.col("es_cancelado").cast("long")).alias("vuelos_cancelados"),
            F.sum(F.col("es_puntual").cast("long")).alias("vuelos_puntuales"),

            F.avg(
                F.when(F.col("es_operado"), F.col("demora_salida_min"))
            ).alias("demora_salida_promedio_min"),

            F.avg(
                F.when(F.col("es_operado"), F.col("demora_llegada_min"))
            ).alias("demora_llegada_promedio_min"),

            F.countDistinct(
                F.when(F.col("es_operado"), F.col("aerolinea_id"))
            ).alias("aerolineas_operadoras"),

            sumar_operados("pasajeros_con_asiento", "pasajeros_con_asiento"),
            sumar_operados("pasajeros_total_a_bordo", "pasajeros_total_a_bordo"),
            sumar_operados("asientos_disponibles", "asientos_ofertados"),
            sumar_operados("pasajeros_no_show", "pasajeros_no_show"),
            sumar_operados("distancia_km", "distancia_total_operada_km"),

            F.sum("combustible_cargado_l").alias("combustible_cargado_l"),
            F.sum("combustible_consumido_l").alias("combustible_consumido_l"),
            F.sum("costo_carga_usd").alias("costo_carga_combustible_usd"),

            F.sum("ingreso_neto_boletos_usd").alias("ingreso_neto_boletos_usd"),
            F.sum("ingreso_equipaje_usd").alias("ingreso_equipaje_usd"),

            sumar_operados("piezas_facturadas", "piezas_facturadas"),
            sumar_operados("piezas_extraviadas", "piezas_extraviadas"),
            sumar_operados("piezas_danadas", "piezas_danadas")
        )
        .withColumn(
            "ingreso_total_usd",
            F.col("ingreso_neto_boletos_usd") + F.col("ingreso_equipaje_usd")
        )
        .withColumn(
            "tasa_cancelacion_pct",
            division_segura("vuelos_cancelados", "vuelos_programados") * 100
        )
        .withColumn(
            "puntualidad_pct",
            division_segura("vuelos_puntuales", "vuelos_operados") * 100
        )
        .withColumn(
            "ocupacion_pct",
            division_segura("pasajeros_con_asiento", "asientos_ofertados") * 100
        )
        .withColumn(
            "combustible_l_por_pasajero",
            division_segura(
                "combustible_consumido_l", "pasajeros_total_a_bordo"
            )
        )
        .withColumn(
            "combustible_l_por_km",
            division_segura(
                "combustible_consumido_l", "distancia_total_operada_km"
            )
        )
        .withColumn(
            "ingreso_usd_por_pasajero",
            division_segura("ingreso_total_usd", "pasajeros_total_a_bordo")
        )
        .withColumn(
            "ingreso_usd_por_vuelo",
            division_segura("ingreso_total_usd", "vuelos_operados")
        )
        .withColumn("fecha_actualizacion", F.current_timestamp())
    )

In [0]:
df_kpi_aerolinea_mes = agregar_indicadores(
    df_base,
    [
        "periodo",
        "anio",
        "mes",
        "aerolinea_id",
        "nombre_aerolinea",
        "pais_sede"
    ]
).select(
    "periodo",
    "anio",
    "mes",
    "aerolinea_id",
    "nombre_aerolinea",
    "pais_sede",
    "vuelos_programados",
    "vuelos_operados",
    "vuelos_cancelados",
    "vuelos_puntuales",
    "tasa_cancelacion_pct",
    "puntualidad_pct",
    "demora_salida_promedio_min",
    "demora_llegada_promedio_min",
    "pasajeros_con_asiento",
    "pasajeros_total_a_bordo",
    "asientos_ofertados",
    "ocupacion_pct",
    "pasajeros_no_show",
    "combustible_cargado_l",
    "combustible_consumido_l",
    "costo_carga_combustible_usd",
    "combustible_l_por_pasajero",
    "ingreso_neto_boletos_usd",
    "ingreso_equipaje_usd",
    "ingreso_total_usd",
    "ingreso_usd_por_pasajero",
    "piezas_facturadas",
    "piezas_extraviadas",
    "piezas_danadas",
    "fecha_actualizacion"
)

In [0]:
df_kpi_ruta_mes = agregar_indicadores(
    df_base,
    [
        "periodo",
        "anio",
        "mes",
        "ruta_id",
        "origen_iata",
        "nombre_aeropuerto_origen",
        "ciudad_origen",
        "pais_origen",
        "destino_iata",
        "nombre_aeropuerto_destino",
        "ciudad_destino",
        "pais_destino",
        "tipo_ruta",
        "distancia_km"
    ]
).select(
    "periodo",
    "anio",
    "mes",
    "ruta_id",
    "origen_iata",
    "nombre_aeropuerto_origen",
    "ciudad_origen",
    "pais_origen",
    "destino_iata",
    "nombre_aeropuerto_destino",
    "ciudad_destino",
    "pais_destino",
    "tipo_ruta",
    "distancia_km",
    "aerolineas_operadoras",
    "vuelos_programados",
    "vuelos_operados",
    "vuelos_cancelados",
    "vuelos_puntuales",
    "tasa_cancelacion_pct",
    "puntualidad_pct",
    "demora_salida_promedio_min",
    "pasajeros_con_asiento",
    "pasajeros_total_a_bordo",
    "asientos_ofertados",
    "ocupacion_pct",
    "distancia_total_operada_km",
    "combustible_consumido_l",
    "combustible_l_por_km",
    "combustible_l_por_pasajero",
    "costo_carga_combustible_usd",
    "ingreso_neto_boletos_usd",
    "ingreso_equipaje_usd",
    "ingreso_total_usd",
    "ingreso_usd_por_vuelo",
    "piezas_facturadas",
    "piezas_extraviadas",
    "fecha_actualizacion"
)

In [0]:
def preparar_para_destino(df, target_table):
    esquema_destino = spark.table(target_table).schema
    columnas_destino = esquema_destino.fieldNames()

    faltantes = set(columnas_destino) - set(df.columns)
    adicionales = set(df.columns) - set(columnas_destino)

    if faltantes or adicionales:
        raise ValueError(
            f"Esquema incompatible en {target_table}. "
            f"Faltantes: {sorted(faltantes)}. "
            f"Adicionales: {sorted(adicionales)}."
        )

    return df.select(
        *[
            F.col(campo.name)
            .cast(campo.dataType)
            .alias(campo.name)
            for campo in esquema_destino.fields
        ]
    )


# Preparar ambos resultados antes de escribir
df_kpi_aerolinea_mes = preparar_para_destino(
    df_kpi_aerolinea_mes,
    target_table_aerolinea
)

df_kpi_ruta_mes = preparar_para_destino(
    df_kpi_ruta_mes,
    target_table_ruta
)

#  ---  Cargar KPI por aerolínea  --- 
target_table = target_table_aerolinea

df_kpi_aerolinea_mes.write \
    .mode("overwrite") \
    .insertInto(target_table)

#  --- Cargar KPI por ruta  --- 
target_table = target_table_ruta

df_kpi_ruta_mes.write \
    .mode("overwrite") \
    .insertInto(target_table)